In [1]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil
import zipfile, subprocess, logging, dotenv, cdsapi
sys.path.append('backend/app/')
from rasterio.io import MemoryFile
from rasterio.features import rasterize
from rasterio.merge import merge
from rasterio.enums import Resampling
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon
from shapely import force_2d
from pyflwdir import dem
from tqdm import tqdm
from pathlib import Path
from owslib.wcs import WebCoverageService
from whitebox.whitebox_tools import WhiteboxTools
from terracatalogueclient import Catalogue
dotenv.load_dotenv()
wtb = WhiteboxTools()
wtb.set_verbose_mode(False)
np.random.seed(42)
from hydromt_wflow import WflowSbmModel
from datetime import datetime
if not hasattr(np, 'bool'): np.bool = np.bool_
logging.getLogger().handlers.clear()

## Child Functions

In [ ]:
def create_forcing(time, ny, nx, values, mask_nan, single_value=True):
    if single_value:
        nt = len(time)
        data = np.zeros((nt, ny, nx), dtype=np.float32)
        for i in range(nt):
            data[i, :, :] = values[i]
        data[:, mask_nan] = 0
    

    return data



In [2]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
lake_path = os.path.join(sample_folder, 'BrusdalLake.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
catchment = gpd.read_file(catchment_path)
# initial_param = [0.25,0.3,50,0,0,500,0,0,100]
raw_dir = os.path.normpath(os.path.join(f'{test_folder}/data/raw'))
if not os.path.exists(raw_dir): os.makedirs(raw_dir)
raw_path = os.path.normpath(os.path.join(raw_dir, "dtm_raw.tif"))
lake = gpd.read_file(lake_path)
NODATA_FLWDR, NODATA_BASIN = 255, 0
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment_UTM = catchment.to_crs(terrain.rio.crs)

## Create a raw terrain that is clipped to catchment

In [3]:
# Clip terrain to catchment
buffer = 10*terrain.rio.resolution()[0] if not terrain.rio.crs.is_geographic else 0.001
catchmentUTM_buffer = catchment_UTM.buffer(buffer)
minx, miny, maxx, maxy = catchmentUTM_buffer.total_bounds
terrain_clipped = terrain.rio.clip_box(minx, miny, maxx, maxy)
terrain_values, nodata = terrain_clipped.values, -9999.0
height, width = terrain_clipped.rio.height, terrain_clipped.rio.width
crs, transform = terrain_clipped.rio.crs, terrain_clipped.rio.transform()
lake_reproj = lake.to_crs(terrain_clipped.rio.crs)
lake_array = rasterize(
    shapes=[geom for geom in lake_reproj.geometry], dtype=np.float32,
    out_shape=(height, width), transform=transform, fill=nodata
)
mask_lake = (lake_array != nodata)
profile = {
    "driver": "GTiff", "count": 1, "dtype": np.float32, "crs": crs,
    "height": height, "width": width, "transform": transform
}
flow_functions.write_geotif(terrain_values, profile, raw_path, nodata)

## Create template hydro data

In [4]:
# Prepare template raster dataset
hydro_dir = os.path.normpath(f'{test_folder}/data/hydro')
if not os.path.exists(hydro_dir): os.makedirs(hydro_dir)
with rasterio.open(raw_path) as src:
    dem_array = src.read(1)
    crs, nodata = src.crs, src.nodata
    transform, profile = src.transform, src.profile
    height, width = src.height, src.width
# Fill depressions
filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
filled_array[mask_lake] = dem_array[mask_lake] # Replace lake elevation
# Create basins
basins_array = flw.basins()
unique, counts = np.unique(basins_array, return_counts=True)
largest_basin_id = unique[np.argmax(counts)]
basins_mask = (basins_array == largest_basin_id)
basins_array[basins_mask], basins_array[~basins_mask] = 1, NODATA_BASIN
elevtn_array = filled_array.copy()
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
gradient_array = filled_array.copy()
gy, gx = np.gradient(gradient_array, dy, dx)
slope_array = np.sqrt(gx**2 + gy**2)
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
# Create stream mask and stream order
stream_mask = (uparea_array > 10)
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
# Create upstream grid
upstream_array = flw.upstream_area(unit='cell')
# Create river width
river = gpd.read_file(river_path).to_crs(crs)
shape = ((geom, value) for geom, value in zip(river.geometry, river["rivwth"]))
rivwth_array = rasterize(
    shapes=shape, out_shape=(profile["height"], profile["width"]),
    transform=transform, fill=nodata, dtype=np.float32
)
files_float = {
    'elevtn.tif': [elevtn_array, nodata, np.float32],
    'flwdir.tif': [flwdir_array, NODATA_FLWDR, np.uint8],
    'lndslp.tif': [slope_array, nodata, np.float32],
    'basins.tif': [basins_array, NODATA_BASIN, np.int32], 
    'uparea.tif': [uparea_array, nodata, np.float32],
    'strord.tif': [strord_array, NODATA_BASIN, np.int16],
    'upgrid.tif': [upstream_array, NODATA_BASIN, np.int32],
    'rivwth.tif': [rivwth_array, nodata, np.float32]
}
profile_writer = profile.copy()
for file, array in files_float.items():
    profile_writer.update({'dtype': array[2]})
    flow_functions.write_geotif(array[0], profile_writer, os.path.join(hydro_dir, file), array[1])

## Process river data

In [5]:
# Create river path
river_dir = os.path.normpath(os.path.join(test_folder, 'data', 'river'))
if not os.path.exists(river_dir): os.makedirs(river_dir)
river_gpkg = os.path.normpath(os.path.join(river_dir, 'river.gpkg'))

In [6]:
# Create river from DEM if it doesn't exist
if not os.path.exists(river_path):
    threshold = 0.1 # threshold in km2
    with rasterio.open(raw_path) as src:
        dem_array = src.read(1).astype(np.float32)
        transform, crs, nodata = src.transform, src.crs, src.nodata
    # mask_nan = np.isnan(dem_array) | (dem_array == nodata)
    filled_array, _ = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
    # flwdir_array = np.where(mask_nan, nodata, flwdir_array)
    flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
    uparea = flw.upstream_area(unit="km2")
    river_mask = uparea > threshold
    features = flw.streams(river_mask)
    gdf = gpd.GeoDataFrame.from_features(features, crs=crs)
    # Clipp river to lake
    clipped_river = gdf.overlay(lake, how='difference')
    clipped_river['lenght'] = clipped_river['geometry'].length
    clipped_river.reset_index(drop=True, inplace=True)
    clipped_river = clipped_river[['geometry']]
    river = clipped_river.reindex(columns=['rivwth', 'rivdph', 'geometry'])
    # Create a random value for each river
    cols = {'rivwth': (0.05, 2), 'rivdph': (1, 5)}
    river_cols = river.columns.drop('geometry', errors='ignore')
    for col in river_cols:
        river[col] = pd.to_numeric(river[col], errors='coerce')
    for col, (low, high) in cols.items():
        river_mask = river[col].isna() | (river[col] == 'None')
        river.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
    river = river[river.is_valid].reset_index(drop=True)
    # Process river
    river["geometry"] = river.geometry.apply(lambda g: force_2d(g))
else:
    river = gpd.read_file(river_path)
    if not 'rivwth' in river.columns or not 'rivdph' in river.columns:
        raise Exception("river.gpkg must have rivwth and rivdph columns")
    river = river[['rivwth', 'rivdph', 'geometry']]
# Write file
river.to_file(river_gpkg, driver='GPKG')
manning_path = r"backend\src\flow_samples\river_manning_mapping.csv"
df = pd.read_csv(manning_path)
df.to_csv(os.path.join(river_dir, 'river_manning.csv'), index=False)

## Prepare forcing data from the customized area

In [4]:
ds = xr.open_dataset(r"test\data\forcing\weather_forcing.nc")
ds

<xarray.Dataset> Size: 3GB
Dimensions:      (time: 217, y: 449, x: 1333)
Coordinates:
  * time         (time) datetime64[ns] 2kB 2025-01-01 ... 2025-01-10
  * y            (y) float64 4kB 6.958e+06 6.958e+06 ... 6.954e+06 6.953e+06
  * x            (x) float64 11kB 5.414e+04 5.415e+04 ... 6.745e+04 6.746e+04
Data variables:
    precip       (time, y, x) float32 520MB ...
    temp         (time, y, x) float32 520MB ...
    kin          (time, y, x) float32 520MB ...
    kout         (time, y, x) float32 520MB ...
    wind         (time, y, x) float32 520MB ...
    press_msl    (time, y, x) float32 520MB ...
    spatial_ref  int32 4B ...

In [5]:
ds.close()

In [11]:
# Prepare forcing data from the global model ARE5
# Source: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download
CDS_url, CDS_key = os.getenv('CDS_URL'), os.getenv('CDS_API_KEY')
config_path = Path.home() / '.cdsapirc'
if not config_path.exists():
    print("Creating .cdsapirc ...")
    config_path.write_text(f"url: {CDS_url}\nkey: {CDS_key}\n", encoding='utf-8')
    print("Created at:", config_path)
catchment_WGS84 = catchment.to_crs('EPSG:4326')
# Setup variables
variables = [
    'total_precipitation', # Precipitation
    '2m_temperature', # Temperature
    '10m_u_component_of_wind', '10m_v_component_of_wind', # Wind
    'surface_pressure',  # Pressure
    'surface_solar_radiation_downwards', # Shortwave radiation
    'surface_thermal_radiation_downwards', # Longwave radiation
]
dataset = 'reanalysis-era5-single-levels'
# start, end = '2025-03-01 00:00:00', '2025-03-02 00:00:00'
# start_time = datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime(end, '%Y-%m-%d %H:%M:%S')
minx, miny, maxx, maxy = catchment_WGS84.total_bounds
area = [round(float(maxy), 2), round(float(minx), 2), round(float(miny), 2) + 10, round(float(maxx), 2) + 10]
# Download ERA5 data monthly
client = cdsapi.Client()
request = {
    'product_type': 'reanalysis', 'variable': [variables[0]],
    'year': ["2025"], 'month': ["01"], 'day': ["01", "02", "03", "04", "05", "06", "07", "08"],
    'time': [f"{h:02d}:00" for h in range(24)], 'area': area,
    'data_format': 'netcdf', 'download_format': 'unarchived'
}
client.retrieve(dataset, request, 'precip.nc')



2026-06-21 23:22:57,995 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-06-21 23:22:57,996 INFO Request ID is f36b06ef-d38e-4c98-bafa-3f1651cfeeba
2026-06-21 23:22:58,089 INFO status has been updated to accepted
2026-06-21 23:23:12,156 INFO status has been updated to running
2026-06-21 23:23:31,336 INFO status has been updated to successful


fdc9595c72437a54572681b849a47435.nc:   0%|          | 0.00/425k [00:00<?, ?B/s]

'precip.nc'

In [12]:
area

[62.5, 6.34, 72.45, 16.59]

In [ ]:
with xr.open_dataset('precip.nc') as ds:
    lat, lon = ds['latitude'].values, ds['longitude'].values
    min_size, vars = min(lat.shape[0], lon.shape[0]), list(ds.data_vars.keys())
    timestamps = pd.to_datetime(ds['valid_time'].values)
    lat, lon = lat[:min_size], lon[:min_size]


# weather_df = pd.DataFrame()

In [23]:
lat, lon, lat.shape[0], lon.shape[0], min(lat.shape[0], lon.shape[0])

(array([72.25, 72.  , 71.75, 71.5 , 71.25, 71.  , 70.75, 70.5 , 70.25,
        70.  , 69.75, 69.5 , 69.25, 69.  , 68.75, 68.5 , 68.25, 68.  ,
        67.75, 67.5 , 67.25, 67.  , 66.75, 66.5 , 66.25, 66.  , 65.75,
        65.5 , 65.25, 65.  , 64.75, 64.5 , 64.25, 64.  , 63.75, 63.5 ,
        63.25, 63.  , 62.75, 62.5 ]),
 array([ 6.5 ,  6.75,  7.  ,  7.25,  7.5 ,  7.75,  8.  ,  8.25,  8.5 ,
         8.75,  9.  ,  9.25,  9.5 ,  9.75, 10.  , 10.25, 10.5 , 10.75,
        11.  , 11.25, 11.5 , 11.75, 12.  , 12.25, 12.5 , 12.75, 13.  ,
        13.25, 13.5 , 13.75, 14.  , 14.25, 14.5 , 14.75, 15.  , 15.25,
        15.5 , 15.75, 16.  , 16.25, 16.5 ]),
 40,
 41,
 40)

In [35]:
ds['tp'].values

array([[[2.4795532e-04, 1.9931793e-04, 1.5354156e-04, ...,
         4.6730042e-05, 2.5749207e-05, 2.7656555e-05],
        [2.2888184e-04, 2.2125244e-04, 1.9931793e-04, ...,
         8.2969666e-05, 6.1988831e-05, 4.4822693e-05],
        [1.7642975e-04, 1.6880035e-04, 1.5544891e-04, ...,
         6.9618225e-05, 6.1035156e-05, 5.1498413e-05],
        ...,
        [6.2942505e-05, 2.5749207e-05, 4.7683716e-06, ...,
         1.9073486e-06, 9.5367432e-07, 9.5367432e-07],
        [4.8637390e-05, 0.0000000e+00, 0.0000000e+00, ...,
         2.8610229e-06, 1.9073486e-06, 9.5367432e-07],
        [0.0000000e+00, 0.0000000e+00, 2.8610229e-06, ...,
         4.7683716e-06, 3.8146973e-06, 3.8146973e-06]],

       [[1.9884109e-04, 1.7929077e-04, 1.5735626e-04, ...,
         2.6226044e-05, 2.7179718e-05, 3.5285950e-05],
        [2.6178360e-04, 2.4080276e-04, 2.0313263e-04, ...,
         3.5285950e-05, 3.6716461e-05, 3.6716461e-05],
        [2.3603439e-04, 2.1457672e-04, 1.7118454e-04, ...,
         4.005

In [21]:
list(ds.data_vars.keys())

['tp']

In [25]:
ds_new

<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 192, latitude: 40, longitude: 41)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 2025-01-01 ... 2025-01-08T23:...
    expver      (valid_time) <U4 3kB ...
  * latitude    (latitude) float64 320B 72.25 72.0 71.75 ... 63.0 62.75 62.5
  * longitude   (longitude) float64 328B 6.5 6.75 7.0 7.25 ... 16.0 16.25 16.5
    number      int64 8B 0
Data variables:
    tp          (valid_time, latitude, longitude) float32 1MB 0.000248 ... 0....
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-21T21:23 GRIB to CDM+CF via cfgrib-0.9.1...

In [27]:
ds_new['tp'].values.shape

(192, 40, 41)

In [18]:
df = pd.DataFrame()
df['lat'], df['lon'] = ds['latitude'].values, ds['longitude'].values
df

ValueError: Length of values (41) does not match length of index (40)

In [10]:
client

In [ ]:

# config_path = Path.home() / '.cdsapirc'
# if not config_path.exists():
#     print("Creating .cdsapirc ...")
#     config_path.write_text(f"url: {CDS_url}\nkey: {CDS_key}\n", encoding='utf-8')
#     print("Created at:", config_path)


# forcing_dir = os.path.join(test_folder, 'data/forcing')
# if not os.path.exists(forcing_dir): os.makedirs(forcing_dir)
# download_dir = os.path.join(test_folder, 'data/forcing/download')
# if not os.path.exists(download_dir): os.makedirs(download_dir)


# # Download ERA5 data monthly
# client = cdsapi.Client()
# for var in variables:
#     current = start_time.replace(day=1)
#     while current <= end_time:
#         year, month = current.year, current.month
#         last_day = calendar.monthrange(year, month)[1]
#         month_start = datetime(year, month, 1)
#         month_end = datetime(year, month, last_day, 23)
#         # Clip by requested range
#         actual_start = max(start_time, month_start)
#         actual_end = min(end_time, month_end)
#         # Days to download
#         days = [f"{d:02d}" for d in range(actual_start.day, actual_end.day + 1)]
#         # Output file
#         out_file = f"{var}_ERA5_{year}_{month:02d}.nc"
#         output = os.path.join(download_dir, out_file)
#         # Skip existing file
#         if os.path.exists(output): os.remove(output)
#         print(f"Downloading: {out_file}")
#         request = {
#             'product_type': 'reanalysis', 'variable': [var],
#             'year': [str(year)], 'month': [f"{month:02d}"], 'day': days,
#             'time': [f"{h:02d}:00" for h in range(24)], 'area': area,
#             'data_format': 'netcdf', 'download_format': 'unarchived'
#         }
#         client.retrieve(dataset, request, output)
#         # Next month
#         current += relativedelta(months=1)

# # Check valid files
# for var in variables:
#     pattern = os.path.join(download_dir, f"{var}_ERA5_*.nc")
#     raw_files = sorted(glob.glob(pattern))
#     # Filter valid files
#     files, bad_files = [], []
#     for f in raw_files:
#         if is_valid_netcdf(f): files.append(f)
#         else: bad_files.append(f)
#     print(f"Valid files '{var}': {len(files)}/{len(raw_files)}")
#     if bad_files:
#         print("Bad files:")
#         for f in bad_files: print(" -", f)

# # Concatenate sub-files
# for var in variables:
#     pattern = os.path.join(download_dir, f"{var}_ERA5_*.nc")
#     raw_files, files = sorted(glob.glob(pattern)), []
#     files = [f for f in raw_files if is_valid_netcdf(f)]
#     if len(files) > 0:
#         ds = xr.open_mfdataset(
#             files, combine='by_coords', parallel=True, chunks={'valid_time':24}
#         )
#         # Remove ERA5 artifact dimension
#         if 'expver' in ds: ds = ds.drop_vars('expver')
#         encoding = {
#             var: {"zlib": True, "complevel": 4, "dtype": "float32"}
#             for var in ds.data_vars
#         }
#         output = pattern.replace('_*', "")
#         print(f"Writing forcing file: {output}")
#         ds.to_netcdf(output, format="NETCDF4", encoding=encoding)
#         ds.close()
#         del ds
#         gc.collect()
# print("DONE:")

# # Merge files
# final_output, datasets = os.path.join(forcing_dir, "my_ear5_forcing.nc"), []
# for var in variables:
#     pattern = os.path.join(download_dir, f"{var}_ERA5.nc")
#     datasets.append(pattern)
# if len(datasets) > 0:
#     datasets_ds = [xr.open_dataset(f) for f in datasets]
#     ds_final = xr.merge(datasets_ds, compat="override", join="outer")
#     encoding = {
#         var: {"zlib": True, "complevel": 4, "dtype": "float32"}
#         for var in ds_final.data_vars
#     }
#     ds_final.to_netcdf(final_output, format="NETCDF4", encoding=encoding)
#     ds_final.close()
#     del ds_final
#     gc.collect()
#     print("DONE:")
# else: print("No files to merge")

# # Change variable name
# rename_dict = {
#     "tp": "precip", "t2m": "temp",
#     "u10": "wind_u", "v10": "wind_v",
#     "ssrd": "radiation",
# }
# final_output = os.path.join(forcing_dir, "my_ear5_forcing.nc")
# final_output_rename = os.path.join(forcing_dir, "my_forcing.nc")
# ds_final = xr.open_dataset(final_output)
# ds_final = ds_final.rename({"longitude": "x", "latitude": "y"})
# ds_final = ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y")
# ds_final = ds_final.rio.write_crs("EPSG:4326")
# ds_rename = ds_final.rename({
#     k: v for k, v in rename_dict.items() if k in ds_final.data_vars
# })
# Unit conversion
columns = weather_df.columns
if "precip" in columns: weather_df["precip_mm"] = weather_df["precip_mm"] * 1000.0
if "temp_C" in columns: weather_df["temp_C"] = weather_df["temp_C"] - 273.15
if "shortwave_Wm2" in columns: weather_df["shortwave_Wm2"] = weather_df["shortwave_Wm2"] / 3600.0
if "longwave_Wm2" in columns: weather_df["longwave_Wm2"] = weather_df["longwave_Wm2"] / 3600.0
weather_df["wind_mps"] = np.sqrt(weather_df["wind_u"]**2 + weather_df["wind_v"]**2)
weather_df["wind_direction"] = (np.degrees(np.arctan2(-weather_df["wind_v"], -weather_df["wind_u"])) + 360) % 360
weather_df = weather_df.drop(columns=["wind_u", "wind_v"])

# ds_rename = ds_rename.sortby("valid_time")
# encoding = {
#     var: {"zlib": True, "complevel": 4, "dtype": "float32"}
#     for var in ds_rename.data_vars
# }
# ds_rename.to_netcdf(final_output_rename, format="NETCDF4", encoding=encoding)

In [11]:
# Read weather data
weather_path = os.path.join(sample_folder, 'weather_2025.csv')
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']
# # Resample
# weather_new = weather_new.resample('1H').interpolate(method='time')

In [12]:
weather_new

,precip_mm,temp_C,shortwave_Wm2,longwave_Wm2,wind_mps,pressure
datetime,,,,,,
2025-01-01 00:00:00,2.447,5.172,0.000,320.125,7.620,1009.946
2025-01-01 01:00:00,2.383,5.172,129.410,384.656,7.993,987.605
2025-01-01 02:00:00,3.192,5.172,250.000,384.239,8.049,997.114
2025-01-01 03:00:00,0.759,5.172,353.553,364.490,7.289,988.409
2025-01-01 04:00:00,1.788,5.172,433.013,322.710,8.959,1013.289
...,...,...,...,...,...,...
2025-01-09 20:00:00,3.320,6.543,0.000,415.198,6.838,995.994
2025-01-09 21:00:00,2.464,6.543,0.000,399.025,5.828,1007.536
2025-01-09 22:00:00,0.587,6.543,0.000,372.547,8.012,1012.469


In [10]:
# Create forcing nc file
time, time_step = weather_new.index.to_numpy(), 'hours'
forcing_dir = os.path.join(test_folder, 'data/forcing')
os.makedirs(forcing_dir, exist_ok=True)
out_path, datasets = os.path.join(forcing_dir, "weather_forcing.nc"), {}
with rasterio.open(raw_path) as src:
    dem_array, crs = src.read(1), src.crs
    transform, nodata = src.transform, src.nodata
mask_nan = np.isnan(dem_array) | (dem_array == nodata)
ny, nx = dem_array.shape[0], dem_array.shape[1]
forcing = {
    'precip': ['precip_mm', 'mm'], 'temp': ['temp_C', 'degC'],
    'kin': ['shortwave_Wm2', 'W/m^2'], 'kout': ['longwave_Wm2', 'W/m^2'],
    'wind': ['wind_mps', 'm/s'], 'press_msl': ['pressure', 'Pa']
}
for var, (col, unit) in forcing.items():
    data = weather_new[col].values.astype(np.float32)
    data_3d = create_forcing(time, ny, nx, data, mask_nan, single_value=True)
    datasets[var] = (('time', 'y', 'x'), data_3d, {'units': unit})
x_coords = transform.c + (np.arange(nx) + 0.5) * transform.a
y_coords = transform.f + (np.arange(ny) + 0.5) * transform.e
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": y_coords, "x": x_coords}
)
ds_final = ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y")
ds_final = ds_final.rio.write_crs(crs)
ds_final["time"].encoding = {
    "units": f"{time_step} since 1900-01-01 00:00:00",
    "calendar": "proleptic_gregorian", "dtype": "float64"
}
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
    for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

## Process land cover

In [11]:
# Create land cover nc file
land_dir = os.path.normpath(os.path.join(test_folder, 'data/landcover'))
if not os.path.exists(land_dir): os.makedirs(land_dir)
terrain_da = rioxarray.open_rasterio(raw_path).squeeze()
with rasterio.open(raw_path) as src:
    nodata, profile, dem_array = src.nodata, src.profile, src.read(1)
profile_writer = profile.copy()
mask_nan = np.isnan(dem_array) | (dem_array == nodata)

In [12]:
# Land cover data from CORINE 2018
corine_path = r"backend\src\flow_samples\landcover\U2018_CLC2018_V2020_20u1.zip"
with zipfile.ZipFile(corine_path, "r") as zip_ref:
    name = os.path.basename(corine_path).replace(".zip", ".tif")
    with zip_ref.open(name) as f:
        with MemoryFile(f.read()) as memfile:
            corine = rioxarray.open_rasterio(memfile).squeeze()
            corine_match = corine.rio.reproject_match(terrain_da, resampling=Resampling.nearest)
land_corine = corine_match.values.astype(np.uint8)
land_corine[mask_nan] = NODATA_FLWDR
path = os.path.join(land_dir, 'corine.tif')
profile_writer.update(dtype=np.uint8)
flow_functions.write_geotif(land_corine, profile_writer, path, NODATA_FLWDR)
# Save look up table
csv_path = r"backend\src\flow_samples\landcover\corine_mapping.csv"
df, lai_name = pd.read_csv(csv_path), "corine"
mask_csv = df['corine'] != 999
df.loc[~mask_csv, 'corine'], df.loc[~mask_csv, 'landuse'] = 48, -999.0
df.loc[mask_csv, 'corine'] = np.arange(1, len(df.loc[mask_csv, 'corine']) + 1).astype(np.uint8)
df.to_csv(os.path.join(land_dir, 'corine.csv'), index=False)
# Compute LAI
lai_path = os.path.join(land_dir, "corine_lai.csv")
flow_functions.create_LAI(land_corine, terrain_da, lai_path, NODATA_FLWDR)

## Process soil data

In [13]:
# Initialize variables
soil_dir = os.path.join(f'{test_folder}/data/soil')
if not os.path.exists(soil_dir): os.makedirs(soil_dir)
catchment_WGS84 = catchmentUTM_buffer.to_crs('EPSG:4326')
min_lon, min_lat, max_lon, max_lat = catchment_WGS84.total_bounds
soil_types = {
    'bulk density': ['BLDFIE_M', 'bd', 1000, np.int16, -32768],
    'clay': ['CLYPPT_M', 'clyppt', 1, np.uint8, 255], 
    'sand': ['SNDPPT_M', 'sndppt', 1, np.uint8, 255],
    'silt': ['SLTPPT_M', 'sltppt', 1, np.uint8, 255],  
    'organic carbon': ['OCDENS_M', 'oc', 1, np.int16, -32768], 
    'pH': ['PHIHOX_M', 'ph', 10, np.uint8, 255]
}
depths = ['sl1', 'sl2', 'sl3', 'sl4', 'sl5', 'sl6', 'sl7']
# Get raster information
ref = rioxarray.open_rasterio(raw_path).squeeze()
with rasterio.open(raw_path) as src:
    profile, transform = src.profile, src.transform
    height, width, dem_array = src.height, src.width, src.read(1)
    nodata, crs = src.nodata, src.crs
# Download soil data 2017 from ISRIC: https://files.isric.org/soilgrids/former/2017-03-10/
base_url = "https://files.isric.org/soilgrids/former/2017-03-10/data/"
# Soil thickness
soil_thickness_url, NODATA_SOIL_THICKNESS = f"{base_url}BDRICM_M_250m_ll.tif", -99999
with rioxarray.open_rasterio(soil_thickness_url) as src:
    data_xr = src.rio.clip_box(minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat)
    data_xr = data_xr.rio.reproject_match(ref, resampling=Resampling.nearest)
    nodata = data_xr.rio.nodata
soil_thickness_array = data_xr[0].values
# Interpolate data
mask_valid = soil_thickness_array != nodata
soil_thickness_values = flow_functions.interpolate_extrapolate(soil_thickness_array, ~mask_valid, True)
soil_thickness_values = soil_thickness_values.astype(np.int32)
soil_thickness_values[mask_lake] = NODATA_SOIL_THICKNESS
soil_thickness_path = os.path.normpath(os.path.join(soil_dir, 'soilthickness.tif'))
profile_writer = profile.copy()
profile_writer.update(dtype=np.int32)
flow_functions.write_geotif(soil_thickness_values, profile_writer, soil_thickness_path, NODATA_SOIL_THICKNESS)
# Download soil data
for item, values in tqdm(soil_types.items(), total=len(soil_types), desc='Downloading 2017 soil data'):
    file, name, scale, dtype, nodata_soil, bulk_density = values[0], values[1], values[2], values[3], values[4], None
    for depth in depths:
        file_url = f'{base_url}{file}_{depth}_250m_ll.tif'
        with rioxarray.open_rasterio(file_url) as src:
            data_xr = src.rio.clip_box(minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat)
            data_xr = data_xr.rio.reproject_match(ref, resampling=Resampling.nearest)
            nodata = data_xr.rio.nodata
        soil_array = data_xr[0].values
        # Interpolate data
        mask_valid = soil_array != nodata
        soil_values = flow_functions.interpolate_extrapolate(soil_array, ~mask_valid, True)
        if name == 'BLDFIE_M': bulk_density = soil_values + 1e-6
        if name == 'OCDENS_M':
            # Convert Organic Carbon Density (kg/m³) to Organic Carbon Content (%)
            soil_values = soil_values / bulk_density * 100
        soil_values = soil_values / scale
        soil_values[mask_lake] = nodata_soil
        path = os.path.join(soil_dir, f'{name}_{depth}.tif')
        profile_writer.update(dtype=dtype)
        flow_functions.write_geotif(soil_values, profile_writer, path, nodata_soil)

## Prepare data using HydroMT

In [5]:
def run_hydromt(mod_path, start, end, step, region, resolution, soil_layers, data_lib, 
    lulc_function='corine', lulc_mapping_fn='corine_mapping', lai_fn='lai_corine'):
    # Prepare model
    if os.path.exists(mod_path): shutil.rmtree(mod_path)
    os.makedirs(mod_path, exist_ok=True)
    model = WflowSbmModel(
        root=mod_path, config_filename='wflow_sbm.toml', data_libs=data_lib, mode='w'
    )
    # Setup configurations
    configs = {
        "time.starttime": datetime.strptime(start, "%Y-%m-%d %H:%M:%S").isoformat(), 
        "time.endtime": datetime.strptime(end, "%Y-%m-%d %H:%M:%S").isoformat(), 
        "time.timestepsecs": step,
        # Reference: https://deltares.github.io/Wflow.jl/dev/model_docs/model_settings.html
        'model.type': 'sbm', # model type: [sbm, sbm_gwf]
        'model.cold_start__flag': True,  # Initialize model with cold (cold_start__flag = true) or warm state
        # Unit cell length of input rasters in lat/lon degree (cell_length_in_meter__flag = false) or in meter
        'model.cell_length_in_meter__flag': False, 'model.reservoir__flag': False, #Include reservoir modelling
        'model.water_mass_balance__flag': False, # Include water mass balance error computations
        'model.snow_gravitational_transport__flag': True, # Include gravitational lateral snow transport
        'model.glacier__flag': False, # Include glacier modelling
        'model.soil_infiltration_reduction__flag': False, # Enable reduction factor applied to the soil infiltration capacity
        'model.snow__flag': True, # Include snow modelling
        # Saturated hydraulic conductivity depth profile for SBM soil model
        # optional, one of ("exponential", "exponential_constant", "layered", "layered_exponential"), default is "exponential"
        'model.saturated_hydraulic_conductivity_profile': 'exponential',
        'model.land_routing': 'kinematic_wave', # Routing approach for overland flow: ["kinematic_wave", "local_inertial"]
        'model.river_routing': 'kinematic_wave', # Routing approach for river flow: ["kinematic_wave", "local_inertial"]
        'model.river_kinematic_wave__time_step': 900, 'model.land_kinematic_wave__time_step': 3600,
        'model.kinematic_wave__adaptive_time_step_flag': False, # Enable kinematic wave adaptive (internal) time stepping
        'output.netcdf_grid.path': 'output.nc', 'output.netcdf_grid.compressionlevel': 2,
        # ========== Output variables ==========
        # Source: https://deltares.github.io/Wflow.jl/previews/PR586/model_docs/parameters_routing.html
        # # Lake variables
        # 'output.netcdf_grid.variables.lake_water__volume': 'lake_volume', # Lake volume (average over timestep), m³
        # 'output.netcdf_grid.variables.lake_water_surface__elevation': 'lake_level', # Lake water level (average over timestep), m
        # 'output.netcdf_grid.variables.lake_water~outgoing__volume_flow_rate': 'lake_outflow', # Outflow of the lake (average over timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.lake_water~incoming__volume_flow_rate': 'lake_inflow', # Inflow into the lake (average over timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.lake_water__evaporation_volume_flux': 'lake_evaporation', # Average actual evaporation over the lake area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.lake_water__precipitation_volume_flux': 'lake_precipitation', # Average precipitation over the lake area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.lake_water__potential_evaporation_volume_flux': 'lake_potential_evaporation', # Average potential evaporation over the lake area	mm Δt⁻¹
        # # Reservoir variables
        # 'output.netcdf_grid.variables.reservoir_water__volume': 'reservoir_volume', # Reservoir volume (average over the timestep)	m³
        # 'output.netcdf_grid.variables.reservoir_water~outgoing__volume_flow_rate': 'reservoir_outflow', # Outflow of the reservoir (average over the timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.reservoir_water~incoming__volume_flow_rate': 'reservoir_inflow', # Inflow into the reservoir (average over the timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.reservoir_water__evaporation_volume_flux': 'reservoir_evaporation', # Average actual evaporation over the reservoir area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.reservoir_water__precipitation_volume_flux': 'reservoir_precipitation', # Average precipitation over the reservoir area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.reservoir_water__potential_evaporation_volume_flux': 'reservoir_potential_evaporation', # Average potential evaporation over the reservoir area	mm Δt⁻¹
        # # River variables (Kinematic wave)
        # 'output.netcdf_grid.variables.river_water__volume_flow_rate': 'river_discharge', # River discharge (average over timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.river_water__depth': 'river_depth', # River depth (average over timestep)	m
        # 'output.netcdf_grid.variables.river_water__volume': 'river_volume', # River volume (average over timestep)	m³
        # 'output.netcdf_grid.variables.river_water_inflow~lateral__volume_flow_rate': 'river_lateral_inflow', # Lateral inflow into the river (average over timestep)	m³ s⁻¹
        # # Overland flow variables
        'output.netcdf_grid.variables.land_surface_water__volume_flow_rate': 'overland_volume_flow', # Overland discharge (average over timestep)	m³ s⁻¹
        'output.netcdf_grid.variables.land_surface_water__depth': 'overland_depth', # Overland depth (average over timestep)	m
        # 'output.netcdf_grid.variables.land_surface_water__volume': 'overland_volume', # Overland volume (average over timestep)	m³
        # 'output.netcdf_grid.variables.land_surface_water__instantaneous_volume_flow_rate': 'overland_discharge_flow', # Discharge overland flow	m³ s⁻¹
        # 'output.netcdf_grid.variables.land_surface_water__instantaneous_depth': 'overland_water_depth_flow', # Water depth overland flow	m
        # # Snow variables
        # 'output.netcdf_grid.variables.snowpack__leq-depth': 'snow_water',  # Liquid-water equivalent of snow pack (SWE)	mm
        # 'output.netcdf_grid.variables.snowpack_meltwater__volume_flux': 'snow_melt',  # Amount of snow melt	mm Δt⁻¹
        # 'output.netcdf_grid.variables.snowpack_water__runoff_volume_flux': 'snow_runoff',  # Runoff from snowpack	mm Δt⁻¹
        # # Glacier variables
        # 'output.netcdf_grid.variables.glacier_ice__melt_volume_flux': 'glacier_melt',  # Melt from the glacier	mm Δt⁻¹
        # # Vegetation variables
        # 'output.netcdf_grid.variables.vegetation_canopy_water__stemflow_volume_flux': 'vegetation_stemflow',  # Stemflow	mm Δt⁻¹
        # 'output.netcdf_grid.variables.vegetation_canopy_water__throughfall_volume_flux': 'vegetation_throughfall',  # Throughfall	mm Δt⁻¹
        # # Soil variables
        # 'output.netcdf_grid.variables.land_surface__evapotranspiration_volume_flux': 'soil_evapotranspiration',  # Total actual evapotranspiration	mm
        # 'output.netcdf_grid.variables.land_water~storage~total__depth': 'soil_storage_total',  # Total water storage (excluding floodplains, lakes and reservoirs)	mm
        # 'output.netcdf_grid.variables.soil_water__infiltration_volume_flux': 'soil_infiltration_volume',  # Actual infiltration into the unsaturated zone	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water__transpiration_volume_flux': 'soil_transpiration_volume',  # Transpiration from vegetation	mm Δt⁻¹
        'output.netcdf_grid.variables.soil_surface_water__runoff_volume_flux': 'soil_runoff',  # Total surface runoff from infiltration and saturation excess	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_surface_water__net_runoff_volume_flux': 'soil_net_runoff',  # Net surface runoff (after open water evaporation)	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_layer_water__volume_fraction': 'soil_water_volume_fraction',  # Volumetric water content per soil layer (including residual water content and saturated zone)
        # 'output.netcdf_grid.variables.soil_layer_water__volume_percentage': 'soil_water_volume_percentage',  # Volumetric water content per soil layer (including residual water content and saturated zone)	%
        # 'output.netcdf_grid.variables.soil_water_root-zone__volume_fraction': 'soil_water_rootzone_volume_fraction',  # Volumetric water content in root zone (including residual water content and saturated zone)
        # 'output.netcdf_grid.variables.soil_water_root-zone__volume_percentage': 'soil_water_rootzone_volume_percentage',  # Volumetric water content in root zone (including residual water content and saturated zone)	%
        # 'output.netcdf_grid.variables.soil_water_root-zone__depth': 'soil_water_rootzone_depth',  # Root water storage in unsaturated and saturated zone (excluding residual water content)	mm
        # 'output.netcdf_grid.variables.soil_water_unsat-zone__depth': 'soil_water_unsatzone_depth',  # Amount of water in the unsaturated store	mm
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__capillary_volume_flux': 'soil_water_satzone_capillary_volume_flux',  # Actual capillary rise	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__recharge_volume_flux': 'soil_water_satzone_recharge_volume_flux',  # Downward flux from unsaturated to saturated zone	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__net_recharge_volume_flux': 'soil_water_satzone_net_recharge_volume_flux',  # Net recharge to saturated zone	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_bottom__leakage_volume_flux': 'soil_water_satzone_leakage_volume_flux',  # Actual leakage from saturated store	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__depth': 'soil_water_satzone_depth',  # Pseudo-water table depth (top of the saturated zone)	mm
    }
    model.setup_config(configs)
    # Setup basemaps: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_basemaps.html
    model.setup_basemaps(
        region=region, hydrography_fn='my_hydro', res=resolution, upscale_method='ihu' # 'ihu', 'eam', 'dmm'
    )
    # Setup rivers: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_rivers.html
    output_names = {
        'river__length': 'river_length', 'river__width': 'river_width', 'river__slope': 'river_slope',
        'river_bank_water__depth': 'river_bank_depth', # Bankfull depth of river, default is 1.0 m
        'river_water_flow__manning_n_parameter': 'river_manning_n', # Manning's roughness, default is 0.036
        'river_bank_water__elevation': 'river_bank_elevation', 'river_location__mask': 'river_mask'
    }
    model.setup_rivers(
        hydrography_fn='my_hydro', river_geom_fn='river_network', 
        river_upa=10, # Minimum upstream area threshold for the river map [km2]
        rivdph_method='powlaw', # 'gvf', 'manning', 'powlaw'
        slope_len=2, #  Length over which the river slope is calculated [km]
        min_rivlen_ratio=0, min_rivdph=1.0, # Minimum river depth [m]
        min_rivwth=30, # Minimum river width [m]
        smooth_len=5000, # Length [m] over which to smooth the output river width and depth
        connectivity=8, river_routing='kinematic_wave', # 'kinematic_wave', 'local_inertial'
        elevtn_map='land_elevation', # Name of the elevation map in the current WflowBaseModel.staticmaps
        output_names=output_names
    )
    model.setup_river_roughness(
        rivman_mapping_fn='river_manning_mapping', # Name of the river manning n map in the current WflowBaseModel.river_maps
        strord_name='meta_streamorder', # Name of the stream order map in the current WflowBaseModel.staticmaps
        output_name='river_manning_n' # Mapping of output variable names.
    )
    # Setup soil maps: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_soilmaps.html
    soil_names = {
        'soil__thickness': 'soil_thickness', 'soil_layer_water__brooks_corey_exponent': 'soil_brooks_corey_c',
        'soil_surface_water__vertical_saturated_hydraulic_conductivity': 'soil_ksat_vertical', 
        'soil_water__residual_volume_fraction': 'soil_theta_r', 'soil_water__saturated_volume_fraction': 'soil_theta_s', 
        'soil_water__vertical_saturated_hydraulic_conductivity_scale_parameter': 'soil_f'
    }
    model.setup_soilmaps(
        soil_fn='soilgrids', ptf_ksatver='brakensiek', # 'brakensiek', 'cosby'
        wflow_thicknesslayers=soil_layers, # Thickness of soil layers [mm] for wflow_sbm soil model
        output_names=soil_names
    )
    model.setup_laimaps_from_lulc_mapping(lulc_fn=lulc_function, lai_mapping_fn=lai_fn)
    # Setup land use maps: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_lulcmaps.html
    lulc_variables = [
        'landuse', 'vegetation_kext', 'land_manning_n', 'soil_compacted_fraction', 
        'vegetation_root_depth', 'vegetation_leaf_storage', 'vegetation_wood_storage', 
        'land_water_fraction', 'vegetation_crop_factor', 'vegetation_feddes_alpha_h1', 
        'vegetation_feddes_h1', 'vegetation_feddes_h2', 'vegetation_feddes_h3_high', 
        'vegetation_feddes_h3_low', 'vegetation_feddes_h4'
    ]
    model.setup_lulcmaps(
        lulc_fn=lulc_function, lulc_mapping_fn=lulc_mapping_fn, lulc_vars=lulc_variables
    )
    # Setup forcing
    model.setup_precip_forcing(precip_fn='weather_forcing')
    model.setup_temp_pet_forcing(
        temp_pet_fn='weather_forcing', pet_method='debruin', # 'debruin', 'makkink', 'penman-monteith_rh_simple', 'penman-monteith_tdew'
        press_correction=True, temp_correction=True, wind_correction=True,
        wind_altitude=10, reproj_method='nearest', fillna_method='nearest',
        dem_forcing_fn='dtm', skip_pet=False
    )
    # === Constant parameters ===
    model.setup_constant_pars(
        subsurface_water__horizontal_to_vertical_saturated_hydraulic_conductivity_ratio = 100,
        snowpack__degree_day_coefficient = 3.75653,
        soil_surface_water__infiltration_reduction_parameter = 0.038,
        vegetation_canopy_water__mean_evaporation_to_mean_precipitation_ratio = 0.11,
        compacted_soil_surface_water__infiltration_capacity = 10,
        soil_water_saturated_zone_bottom__max_leakage_volume_flux = 0,
        soil_wet_root__sigmoid_function_shape_parameter = -500,
        atmosphere_air__snowfall_temperature_threshold = 0,
        atmosphere_air__snowfall_temperature_interval = 2,
        snowpack__melting_temperature_threshold = 0,
        snowpack__liquid_water_holding_capacity = 0.1,
        glacier_ice__degree_day_coefficient = 3,
        glacier_firn_accumulation__snowpack_dry_snow_leq_depth_fraction = 0.001,
        glacier_ice__melting_temperature_threshold = 0
    )
    # === Cold states ===
    model.setup_cold_states()
    # === Write model ===
    model.write(
        grid_filename='static_grid.nc', geoms_folder='staticgeoms', 
        forcing_filename='weather_forcing.nc', states_filename='output_state.nc'
    )
    return model

In [ ]:
# # Check parameters
# nan_vars = []
# with xr.open_dataset(r"test\model_new\static_grid.nc") as ds:
#     for item in ds.data_vars:
#         if np.unique(ds[item].values).size == 1 and np.isnan(np.unique(ds[item].values)[0]):
#             nan_vars.append(item)
# if len(nan_vars) > 0: raise ValueError(f"NaN values found in {nan_vars}")
# else: print("No NaN values found")

mod_path = 'model'
if not os.path.exists(mod_path): os.makedirs(mod_path)
start, end, step = "2025-01-01 00:00:00", "2025-01-10 00:00:00", 3600
# lulc_function, lulc_mapping_fn, lai_fn = 'esa_worldcover', 'esa_worldcover_mapping', 'lai_esa'
# Source: https://deltares.github.io/hydromt/v0.8.0/_generated/hydromt.workflows.basin_mask.parse_region.html
region = {'subbasin': [55010.153, 6955129.102]}
soil_layers = [50, 100, 150, 300, 400, 600]
data_lib = ["./test/config.yml"]

model = run_hydromt(
    mod_path, start, end, step, region, 10, soil_layers, data_lib, #lulc_function, lulc_mapping_fn, lai_fn
)

2026-06-20 21:15:29,589 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-06-20 21:15:29,598 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-06-20 21:15:29,598 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from c:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-06-20 21:15:29,630 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-20 21:15:29,631 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from c:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-06-20 21:15:29,634 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Preparing base hydrography basemaps.
2026-06-20 21:15:29,635 - hydromt.data_catalog.sources.data_source - data_source - INFO - Reading my_hydro RasterDataset data from d:\Programmin

## Run simulation

In [8]:
wflow_exe = r"D:\Programming_Codes\Hydro-AI-Platform\backend\softs\wflow 1.0.2\wflow_cli\bin\wflow_cli.exe"
cwd = r"D:\Programming_Codes\Hydro-AI-Platform\model"
toml_path = r"D:\Programming_Codes\Hydro-AI-Platform\model\wflow_sbm.toml"
cmd = [wflow_exe, toml_path]
process = subprocess.Popen(
    cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout: print(line, end='')
process.wait()
print('Return code:', process.returncode)

[ Info: Wflow version v1.0.2
[ Info: Initialize model variables for model type sbm.
â”Œ Info: Cyclic parameters are provided by
â”” D:\Programming_Codes\Hydro-AI-Platform\model\static_grid.nc.
â”Œ Info: Forcing parameters are provided by
â”” D:\Programming_Codes\Hydro-AI-Platform\model\weather_forcing.nc.
â”Œ Info: Set atmosphere_water__precipitation_volume_flux using netCDF variable
â”” precip as forcing parameter.
â”Œ Info: Set atmosphere_air__temperature using netCDF variable temp as forcing
â”” parameter.
â”Œ Info: Set land_surface_water__potential_evaporation_volume_flux using
â”” netCDF variable pet as forcing parameter.
â”Œ Info: Set vegetation__leaf_area_index using netCDF variable
â”” vegetation_leaf_area_index as cyclic parameter, with 12 timesteps.
â”Œ Info: General model settings
â”‚   snow = true
â”‚   gravitational_snow_transport = true
â”‚   glacier = false
â”‚   reservoirs = false
â”‚   pits = false
â””   water_demand = false
[ Info: Set subbasin_location__count using n